In [ ]:
#@title Language Selection / Выбор языка
#@markdown Choose your language / Выберите язык:
LANGUAGE = "English"  #@param ["English", "Russian"]

TEXTS = {
    "English": {
        "lang_selected": "Language set to English.",
        "mount_drive": "Mounting Google Drive...",
        "drive_mounted": "Google Drive mounted successfully.",
        "output_dir_created": "Output directory created: {}",
        "hardware_info": "Hardware detected: {} GPU, {} CPU cores, {:.1f} GB RAM",
        "no_gpu": "No GPU detected. Using CPU-only mode.",
        "gpu_detected": "GPU detected: {}",
        "choose_input": "Choose image input method:",
        "option_upload": "1 - Upload from PC",
        "option_drive": "2 - Load from Google Drive",
        "enter_choice": "Enter 1 or 2: ",
        "enter_path": "Enter the full path to the image in Google Drive: ",
        "file_not_found": "File not found: {}",
        "unsupported_format": "Unsupported format. Use PNG, JPG, or BMP.",
        "image_loaded": "Image loaded: {}x{} pixels",
        "choose_preset": "Choose a quality preset:",
        "preset_selected": "Preset selected: {}",
        "custom_prompt": "Enter custom value for {} (or press Enter to keep {}): ",
        "settings_summary": "Settings: stopAt={}, randomSamples={}, mutatedSamples={}, maxResolution={}, posterizeLevels={}",
        "starting": "Starting geometry generation...",
        "progress": "Shape {}/{} | Error reduction: {:.4f} | Elapsed: {:.1f}s | Speed: {:.1f} shapes/s",
        "checkpoint_saved": "Checkpoint saved: {} ({} shapes)",
        "generation_complete": "Generation complete! {} shapes in {:.1f}s ({:.1f} shapes/s)",
        "saving_output": "Saving output to Google Drive...",
        "output_saved": "Output saved to: {}",
        "preview_title": "Generated Result Preview",
        "progress_bar_desc": "Generating shapes",
        "batch_size_info": "Auto batch size: {} (GPU: {})",
    },
    "Russian": {
        "lang_selected": "Язык установлен: Русский.",
        "mount_drive": "Подключение Google Drive...",
        "drive_mounted": "Google Drive подключен успешно.",
        "output_dir_created": "Папка для вывода создана: {}",
        "hardware_info": "Оборудование: {} GPU, {} ядер CPU, {:.1f} ГБ RAM",
        "no_gpu": "GPU не обнаружен. Используется режим только CPU.",
        "gpu_detected": "GPU обнаружен: {}",
        "choose_input": "Выберите способ загрузки изображения:",
        "option_upload": "1 - Загрузить с компьютера",
        "option_drive": "2 - Загрузить из Google Drive",
        "enter_choice": "Введите 1 или 2: ",
        "enter_path": "Введите полный путь к изображению в Google Drive: ",
        "file_not_found": "Файл не найден: {}",
        "unsupported_format": "Неподдерживаемый формат. Используйте PNG, JPG или BMP.",
        "image_loaded": "Изображение загружено: {}x{} пикселей",
        "choose_preset": "Выберите пресет качества:",
        "preset_selected": "Выбран пресет: {}",
        "custom_prompt": "Введите значение для {} (или Enter для {}): ",
        "settings_summary": "Настройки: stopAt={}, randomSamples={}, mutatedSamples={}, maxResolution={}, posterizeLevels={}",
        "starting": "Начинается генерация геометрии...",
        "progress": "Фигура {}/{} | Снижение ошибки: {:.4f} | Время: {:.1f}с | Скорость: {:.1f} фигур/с",
        "checkpoint_saved": "Контрольная точка сохранена: {} ({} фигур)",
        "generation_complete": "Генерация завершена! {} фигур за {:.1f}с ({:.1f} фигур/с)",
        "saving_output": "Сохранение результатов в Google Drive...",
        "output_saved": "Результат сохранен в: {}",
        "preview_title": "Предварительный просмотр результата",
        "progress_bar_desc": "Генерация фигур",
        "batch_size_info": "Авто размер батча: {} (GPU: {})",
    },
}

T = TEXTS[LANGUAGE]
print(T["lang_selected"])


In [ ]:
#@title Setup & Drive Mount
import os
import subprocess

from google.colab import drive

print(T["mount_drive"])
drive.mount("/content/drive")
print(T["drive_mounted"])

OUTPUT_DIR = "/content/drive/MyDrive/output/"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(T["output_dir_created"].format(OUTPUT_DIR))

# Detect hardware
import multiprocessing
cpu_cores = multiprocessing.cpu_count()

try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024**3)
except ImportError:
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / (1024**3)

GPU_AVAILABLE = False
GPU_NAME = "None"
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                           capture_output=True, text=True, timeout=10)
    if result.returncode == 0 and result.stdout.strip():
        GPU_NAME = result.stdout.strip().split("\n")[0]
        GPU_AVAILABLE = True
        print(T["gpu_detected"].format(GPU_NAME))
except (FileNotFoundError, subprocess.TimeoutExpired):
    pass

if not GPU_AVAILABLE:
    print(T["no_gpu"])

print(T["hardware_info"].format(GPU_NAME, cpu_cores, ram_gb))


In [ ]:
#@title Image Input
from PIL import Image
from IPython.display import display
import numpy as np

SUPPORTED_FORMATS = (".png", ".jpg", ".jpeg", ".bmp")

print(T["choose_input"])
print(T["option_upload"])
print(T["option_drive"])
choice = input(T["enter_choice"]).strip()

if choice == "1":
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file uploaded.")
    filename = list(uploaded.keys())[0]
    image_path = os.path.join("/content", filename)
    with open(image_path, "wb") as f:
        f.write(uploaded[filename])
else:
    image_path = input(T["enter_path"]).strip()
    if not os.path.isfile(image_path):
        raise FileNotFoundError(T["file_not_found"].format(image_path))

ext = os.path.splitext(image_path)[1].lower()
if ext not in SUPPORTED_FORMATS:
    raise ValueError(T["unsupported_format"])

target_image = Image.open(image_path).convert("RGB")
img_width, img_height = target_image.size
print(T["image_loaded"].format(img_width, img_height))
display(target_image)


In [ ]:
#@title Settings / Preset Selection
PRESETS = {
    "1": {
        "name": "Extremely Fast",
        "stopAt": 500,
        "randomSamples": 30000,
        "mutatedSamples": 1000,
        "maxResolution": 600,
        "posterizeLevels": 10,
        "errorGridSize": 64,
        "saveAt": [100, 200, 300, 400, 500],
    },
    "2": {
        "name": "Fast",
        "stopAt": 1000,
        "randomSamples": 60000,
        "mutatedSamples": 2000,
        "maxResolution": 900,
        "posterizeLevels": 10,
        "errorGridSize": 64,
        "saveAt": [250, 500, 750, 1000],
    },
    "3": {
        "name": "Balanced",
        "stopAt": 1800,
        "randomSamples": 120000,
        "mutatedSamples": 5000,
        "maxResolution": 1400,
        "posterizeLevels": 20,
        "errorGridSize": 64,
        "saveAt": [300, 600, 900, 1200, 1500, 1800],
    },
    "4": {
        "name": "Slow",
        "stopAt": 2500,
        "randomSamples": 220000,
        "mutatedSamples": 8000,
        "maxResolution": 2000,
        "posterizeLevels": 50,
        "errorGridSize": 64,
        "saveAt": [500, 1000, 1500, 2000, 2500],
    },
    "5": {
        "name": "Super Slow",
        "stopAt": 3000,
        "randomSamples": 350000,
        "mutatedSamples": 12000,
        "maxResolution": 2400,
        "posterizeLevels": 100,
        "errorGridSize": 64,
        "saveAt": [500, 1000, 1500, 2000, 2500, 3000],
    },
}

print(T["choose_preset"])
for key, preset in PRESETS.items():
    print(f"  {key}. {preset['name']} ({preset['stopAt']} shapes, {preset['randomSamples']} samples)")

preset_choice = input(T["enter_choice"]).strip()
if preset_choice not in PRESETS:
    preset_choice = "3"

settings = dict(PRESETS[preset_choice])
print(T["preset_selected"].format(settings["name"]))

# Allow custom overrides
for param in ["stopAt", "randomSamples", "mutatedSamples", "maxResolution"]:
    val = input(T["custom_prompt"].format(param, settings[param])).strip()
    if val:
        settings[param] = int(val)

print(T["settings_summary"].format(
    settings["stopAt"], settings["randomSamples"],
    settings["mutatedSamples"], settings["maxResolution"],
    settings["posterizeLevels"]
))


In [ ]:
#@title Geometrize Engine
import numpy as np
import time
import math

# Shape type constants
RECTANGLE = 1
ROTATED_ELLIPSE = 16

# ---------- GPU Setup ----------
USE_GPU = False
if GPU_AVAILABLE:
    try:
        import cupy as cp
        # Quick test to verify CuPy works
        _test = cp.zeros(1)
        del _test
        USE_GPU = True
        print("GPU acceleration enabled with CuPy")
    except Exception as e:
        print(f"GPU setup failed ({e}), falling back to CPU.")
        USE_GPU = False

if not USE_GPU:
    print("Using CPU-only mode with numpy vectorized operations.")


def posterize_color(r, g, b, levels):
    """Quantize color to the given number of posterization levels."""
    if levels <= 1:
        return int(r), int(g), int(b)
    step = 255.0 / (levels - 1)
    pr = int(round(round(r / step) * step))
    pg = int(round(round(g / step) * step))
    pb = int(round(round(b / step) * step))
    return max(0, min(255, pr)), max(0, min(255, pg)), max(0, min(255, pb))


def get_shape_mask(cx, cy, hw, hh, rot_deg, shape_type, img_h, img_w):
    """Get pixel coordinates inside a shape. Returns (ys, xs) arrays."""
    x_min = max(0, cx - hw)
    x_max = min(img_w, cx + hw)
    y_min = max(0, cy - hh)
    y_max = min(img_h, cy + hh)

    if x_min >= x_max or y_min >= y_max:
        return np.array([], dtype=np.int32), np.array([], dtype=np.int32)

    ys, xs = np.mgrid[y_min:y_max, x_min:x_max]
    ys_flat = ys.ravel()
    xs_flat = xs.ravel()

    dx = xs_flat - cx
    dy = ys_flat - cy

    if rot_deg != 0:
        rad = rot_deg * np.pi / 180.0
        cos_a = np.cos(rad)
        sin_a = np.sin(rad)
        rx = dx * cos_a + dy * sin_a
        ry = -dx * sin_a + dy * cos_a
    else:
        rx = dx.astype(np.float64)
        ry = dy.astype(np.float64)

    if shape_type == RECTANGLE:
        mask = (np.abs(rx) <= hw) & (np.abs(ry) <= hh)
    else:  # ROTATED_ELLIPSE
        if hw > 0 and hh > 0:
            mask = (rx * rx) / (hw * hw) + (ry * ry) / (hh * hh) <= 1.0
        else:
            return np.array([], dtype=np.int32), np.array([], dtype=np.int32)

    return ys_flat[mask], xs_flat[mask]


def compute_optimal_color(target, canvas, ys, xs, posterize_levels):
    """Compute the optimal color for the shape area."""
    if len(ys) == 0:
        return 128, 128, 128

    target_pixels = target[ys, xs]  # (N, 3)
    avg_r = np.mean(target_pixels[:, 0])
    avg_g = np.mean(target_pixels[:, 1])
    avg_b = np.mean(target_pixels[:, 2])

    return posterize_color(avg_r, avg_g, avg_b, posterize_levels)


def compute_error_reduction(target, canvas, ys, xs, r, g, b):
    """Compute how much painting this shape reduces pixel error."""
    if len(ys) == 0:
        return 0.0

    target_pixels = target[ys, xs].astype(np.float64)
    canvas_pixels = canvas[ys, xs].astype(np.float64)

    # Current error in shape region
    old_diff = canvas_pixels - target_pixels
    old_error = np.sum(old_diff * old_diff)

    # New error with proposed color
    new_color = np.array([r, g, b], dtype=np.float64)
    new_diff = new_color - target_pixels
    new_error = np.sum(new_diff * new_diff)

    return float(old_error - new_error)


def generate_random_candidates(n, img_w, img_h, rng):
    """Generate n random candidate shapes."""
    types = rng.choice([RECTANGLE, ROTATED_ELLIPSE], size=n)
    cx = rng.integers(0, img_w, size=n)
    cy = rng.integers(0, img_h, size=n)
    # Half-widths and half-heights (size range: 2 to max_dim/4)
    max_hw = max(4, img_w // 4)
    max_hh = max(4, img_h // 4)
    hw = rng.integers(2, max_hw, size=n)
    hh = rng.integers(2, max_hh, size=n)
    rot = rng.integers(0, 360, size=n)
    return types, cx, cy, hw, hh, rot


def evaluate_candidates_gpu(target_gpu, canvas_gpu, types, cx, cy, hw, hh, rot,
                            img_h, img_w, posterize_levels):
    """Evaluate all candidates on GPU using CuPy vectorized tensor operations.
    Creates mask tensors of shape (batch, H, W) and uses element-wise operations
    to compute everything in parallel across all pixels."""
    n = len(types)
    all_scores = np.zeros(n, dtype=np.float64)
    all_colors = np.zeros((n, 3), dtype=np.int32)

    # Sub-batch to manage GPU memory
    sub_batch_size = max(16, min(n, int(4e9 / (img_h * img_w * 4))))

    for start in range(0, n, sub_batch_size):
        end = min(start + sub_batch_size, n)
        batch_n = end - start

        b_cx = cp.asarray(cx[start:end], dtype=cp.float32)
        b_cy = cp.asarray(cy[start:end], dtype=cp.float32)
        b_hw = cp.asarray(hw[start:end], dtype=cp.float32)
        b_hh = cp.asarray(hh[start:end], dtype=cp.float32)
        b_rot = cp.asarray(rot[start:end], dtype=cp.float32)
        b_types = types[start:end]

        # Coordinate grids: (H, W)
        yy = cp.arange(img_h, dtype=cp.float32).reshape(img_h, 1)
        xx = cp.arange(img_w, dtype=cp.float32).reshape(1, img_w)

        # dx, dy: (batch, H, W)
        dx = xx[None, :, :] - b_cx[:, None, None]
        dy = yy[None, :, :] - b_cy[:, None, None]

        # Rotation: (batch, 1, 1)
        rad = b_rot[:, None, None] * (cp.pi / 180.0)
        cos_a = cp.cos(rad)
        sin_a = cp.sin(rad)
        rx = dx * cos_a + dy * sin_a
        ry = -dx * sin_a + dy * cos_a

        # Masks
        hw_f = b_hw[:, None, None]
        hh_f = b_hh[:, None, None]

        rect_mask = (cp.abs(rx) <= hw_f) & (cp.abs(ry) <= hh_f)
        hw_safe = cp.maximum(hw_f, 1.0)
        hh_safe = cp.maximum(hh_f, 1.0)
        ellipse_mask = (rx**2 / hw_safe**2 + ry**2 / hh_safe**2) <= 1.0

        is_rect = cp.asarray((b_types == RECTANGLE))[:, None, None]
        masks = cp.where(is_rect, rect_mask, ellipse_mask)  # (B, H, W)

        # Free intermediate arrays
        del dx, dy, rx, ry, rect_mask, ellipse_mask, rad, cos_a, sin_a

        # Pixel counts
        pixel_counts = masks.sum(axis=(1, 2))  # (B,)

        # Masked target sum for color computation
        masks_f = masks.astype(cp.float32)  # (B, H, W)
        # target_gpu is (H, W, 3) - expand to (1, H, W, 3)
        # masks_f is (B, H, W) - expand to (B, H, W, 1)
        masked_target = target_gpu[None, :, :, :] * masks_f[:, :, :, None]  # (B, H, W, 3)
        color_sums = masked_target.sum(axis=(1, 2))  # (B, 3)
        del masked_target

        # Average colors
        valid_counts = cp.maximum(pixel_counts, 1.0)[:, None]
        avg_colors = color_sums / valid_counts  # (B, 3)

        # Posterize
        if posterize_levels > 1:
            step = 255.0 / (posterize_levels - 1)
            avg_colors = cp.round(cp.round(avg_colors / step) * step)
            avg_colors = cp.clip(avg_colors, 0, 255)

        # Default for empty masks
        valid_mask = (pixel_counts > 0)[:, None]
        avg_colors = cp.where(valid_mask, avg_colors, 128.0)

        # Error computation
        # old error: sum((canvas - target)^2) in mask
        old_diff = (canvas_gpu[None, :, :, :] - target_gpu[None, :, :, :]) * masks_f[:, :, :, None]
        old_error = (old_diff * old_diff).sum(axis=(1, 2, 3))  # (B,)
        del old_diff

        # new error: sum((color - target)^2) in mask
        new_diff = (avg_colors[:, None, None, :] - target_gpu[None, :, :, :]) * masks_f[:, :, :, None]
        new_error = (new_diff * new_diff).sum(axis=(1, 2, 3))  # (B,)
        del new_diff, masks_f

        scores = old_error - new_error

        all_scores[start:end] = cp.asnumpy(scores)
        all_colors[start:end] = cp.asnumpy(avg_colors).astype(np.int32)

        del masks, scores
        cp.get_default_memory_pool().free_all_blocks()

    return all_scores, all_colors


def evaluate_candidates_cpu(target, canvas, types, cx, cy, hw, hh, rot,
                            img_h, img_w, posterize_levels):
    """Evaluate all candidates on CPU using numpy. Returns (scores, colors)."""
    n = len(types)
    scores = np.zeros(n)
    colors = np.zeros((n, 3), dtype=np.int32)

    for i in range(n):
        ys, xs = get_shape_mask(int(cx[i]), int(cy[i]), int(hw[i]), int(hh[i]),
                                int(rot[i]), int(types[i]), img_h, img_w)
        if len(ys) == 0:
            continue
        r, g, b = compute_optimal_color(target, canvas, ys, xs, posterize_levels)
        score = compute_error_reduction(target, canvas, ys, xs, r, g, b)
        scores[i] = score
        colors[i] = [r, g, b]

    return scores, colors


def mutate_candidate(cx, cy, hw, hh, rot, shape_type, img_w, img_h, rng):
    """Apply a small random mutation to a candidate shape."""
    mutation_type = rng.integers(0, 4)
    new_cx, new_cy, new_hw, new_hh, new_rot = cx, cy, hw, hh, rot

    if mutation_type == 0:  # Move
        new_cx = max(0, min(img_w - 1, cx + rng.integers(-20, 21)))
        new_cy = max(0, min(img_h - 1, cy + rng.integers(-20, 21)))
    elif mutation_type == 1:  # Resize width
        new_hw = max(2, min(img_w // 2, hw + rng.integers(-10, 11)))
    elif mutation_type == 2:  # Resize height
        new_hh = max(2, min(img_h // 2, hh + rng.integers(-10, 11)))
    elif mutation_type == 3:  # Rotate
        new_rot = (rot + rng.integers(-30, 31)) % 360

    return new_cx, new_cy, new_hw, new_hh, new_rot


def hill_climb(target_gpu, canvas_gpu, target, canvas, shape_type, cx, cy, hw, hh, rot,
               best_score, best_color, n_mutations, img_h, img_w, posterize_levels, rng):
    """Hill-climb the best candidate with batched mutations.
    When GPU is available, evaluates all mutations in parallel using CuPy."""
    cur_cx, cur_cy, cur_hw, cur_hh, cur_rot = cx, cy, hw, hh, rot
    cur_score = best_score
    cur_color = best_color

    if USE_GPU:
        # Batch all mutations and evaluate on GPU in rounds
        n_rounds = min(8, max(1, n_mutations // 500))
        mutations_per_round = max(64, n_mutations // n_rounds)

        for _ in range(n_rounds):
            # Generate batch of mutations from current best
            m_cx = np.clip(cur_cx + rng.integers(-20, 21, size=mutations_per_round), 0, img_w - 1).astype(np.int32)
            m_cy = np.clip(cur_cy + rng.integers(-20, 21, size=mutations_per_round), 0, img_h - 1).astype(np.int32)
            m_hw = np.clip(cur_hw + rng.integers(-10, 11, size=mutations_per_round), 2, img_w // 2).astype(np.int32)
            m_hh = np.clip(cur_hh + rng.integers(-10, 11, size=mutations_per_round), 2, img_h // 2).astype(np.int32)
            m_rot = ((cur_rot + rng.integers(-30, 31, size=mutations_per_round)) % 360).astype(np.int32)
            m_types = np.full(mutations_per_round, shape_type, dtype=np.int32)

            # Randomly select which parameter to mutate for each candidate
            mutation_type = rng.integers(0, 5, size=mutations_per_round)
            # Keep original values for non-mutated parameters
            keep_cx = np.full(mutations_per_round, cur_cx, dtype=np.int32)
            keep_cy = np.full(mutations_per_round, cur_cy, dtype=np.int32)
            keep_hw = np.full(mutations_per_round, cur_hw, dtype=np.int32)
            keep_hh = np.full(mutations_per_round, cur_hh, dtype=np.int32)
            keep_rot = np.full(mutations_per_round, cur_rot, dtype=np.int32)

            # Apply mutations selectively
            final_cx = np.where((mutation_type == 0) | (mutation_type == 4), m_cx, keep_cx)
            final_cy = np.where((mutation_type == 0) | (mutation_type == 4), m_cy, keep_cy)
            final_hw = np.where(mutation_type == 1, m_hw, keep_hw)
            final_hh = np.where(mutation_type == 2, m_hh, keep_hh)
            final_rot = np.where(mutation_type == 3, m_rot, keep_rot)

            # Evaluate on GPU
            scores, colors = evaluate_candidates_gpu(
                target_gpu, canvas_gpu, m_types, final_cx, final_cy, final_hw, final_hh, final_rot,
                img_h, img_w, posterize_levels
            )

            best_idx = np.argmax(scores)
            if scores[best_idx] > cur_score:
                cur_cx = int(final_cx[best_idx])
                cur_cy = int(final_cy[best_idx])
                cur_hw = int(final_hw[best_idx])
                cur_hh = int(final_hh[best_idx])
                cur_rot = int(final_rot[best_idx])
                cur_score = float(scores[best_idx])
                cur_color = (int(colors[best_idx][0]), int(colors[best_idx][1]), int(colors[best_idx][2]))
    else:
        # CPU fallback: sequential mutations (original approach)
        for _ in range(n_mutations):
            m_cx, m_cy, m_hw, m_hh, m_rot = mutate_candidate(
                cur_cx, cur_cy, cur_hw, cur_hh, cur_rot, shape_type, img_w, img_h, rng
            )
            ys, xs = get_shape_mask(m_cx, m_cy, m_hw, m_hh, m_rot, shape_type, img_h, img_w)
            if len(ys) == 0:
                continue
            r, g, b = compute_optimal_color(target, canvas, ys, xs, posterize_levels)
            score = compute_error_reduction(target, canvas, ys, xs, r, g, b)
            if score > cur_score:
                cur_cx, cur_cy, cur_hw, cur_hh, cur_rot = m_cx, m_cy, m_hw, m_hh, m_rot
                cur_score = score
                cur_color = (r, g, b)

    return shape_type, cur_cx, cur_cy, cur_hw, cur_hh, cur_rot, cur_score, cur_color



def paint_shape(canvas, shape_type, cx, cy, hw, hh, rot, r, g, b, img_h, img_w):
    """Paint a shape onto the canvas."""
    ys, xs = get_shape_mask(cx, cy, hw, hh, rot, shape_type, img_h, img_w)
    if len(ys) > 0:
        canvas[ys, xs] = [r, g, b]


def resize_image(image, max_resolution):
    """Resize image so the longest side is max_resolution."""
    w, h = image.size
    if max(w, h) <= max_resolution:
        return image
    scale = max_resolution / max(w, h)
    new_w = int(w * scale)
    new_h = int(h * scale)
    return image.resize((new_w, new_h), Image.LANCZOS)


def build_output_json(shapes, img_w, img_h):
    """Build the output JSON structure matching geometry_json.py format."""
    output_shapes = []
    # Background rectangle (first shape)
    bg_color = shapes[0]["color"] if shapes else [0, 0, 0, 255]
    output_shapes.append({
        "type": RECTANGLE,
        "data": [0, 0, img_w, img_h],
        "color": bg_color,
        "score": 0,
    })
    # Remaining shapes
    for shape in shapes[1:]:
        output_shapes.append(shape)
    return {"shapes": output_shapes}


print("Geometrize engine loaded successfully.")


In [ ]:
#@title Run Generation
import json
import time
from tqdm.notebook import tqdm
import sys

# Resize image to maxResolution
processed_image = resize_image(target_image, settings["maxResolution"])
proc_w, proc_h = processed_image.size
target_arr = np.array(processed_image, dtype=np.uint8)  # (H, W, 3)
img_h, img_w = target_arr.shape[:2]

print(T["starting"])
print(f"  Processing at {img_w}x{img_h} resolution")

# Initialize canvas with the average color of the target image
avg_color = target_arr.mean(axis=(0, 1)).astype(np.uint8)
canvas = np.full_like(target_arr, avg_color)

# Store generated shapes
generated_shapes = []
bg_r, bg_g, bg_b = int(avg_color[0]), int(avg_color[1]), int(avg_color[2])
generated_shapes.append({
    "type": RECTANGLE,
    "data": [0, 0, img_w, img_h],
    "color": [bg_r, bg_g, bg_b, 255],
    "score": 0,
})

rng = np.random.default_rng(42)
start_time = time.time()

stop_at = settings["stopAt"]
random_samples = settings["randomSamples"]
mutated_samples = settings["mutatedSamples"]
posterize_levels = settings["posterizeLevels"]
save_at_points = set(settings["saveAt"])

# Auto-tune BATCH_SIZE based on GPU memory
if USE_GPU:
    BATCH_SIZE = min(512, int(4e9 / (img_h * img_w * 4)))
    BATCH_SIZE = max(16, min(BATCH_SIZE, random_samples))
else:
    BATCH_SIZE = max(2000, cpu_cores * 500)
    BATCH_SIZE = min(BATCH_SIZE, random_samples)
print(T["batch_size_info"].format(BATCH_SIZE, GPU_AVAILABLE))

# Initialize GPU arrays
target_gpu = None
canvas_gpu = None
if USE_GPU:
    target_gpu = cp.asarray(target_arr, dtype=cp.float32)
    canvas_gpu = cp.asarray(canvas, dtype=cp.float32)

pbar = tqdm(range(1, stop_at + 1), desc=T["progress_bar_desc"], unit="shape", mininterval=0, dynamic_ncols=True)
sys.stdout.flush()
for shape_idx in pbar:
    best_score = -1e30
    best_shape = None
    best_color = (128, 128, 128)

    # Evaluate random candidates in batches
    remaining = random_samples
    while remaining > 0:
        batch_n = min(BATCH_SIZE, remaining)
        types, cx, cy, hw, hh, rot = generate_random_candidates(batch_n, img_w, img_h, rng)

        if USE_GPU and batch_n >= 16:
            scores, colors = evaluate_candidates_gpu(
                target_gpu, canvas_gpu, types, cx, cy, hw, hh, rot,
                img_h, img_w, posterize_levels
            )
        else:
            scores, colors = evaluate_candidates_cpu(
                target_arr, canvas, types, cx, cy, hw, hh, rot,
                img_h, img_w, posterize_levels
            )

        batch_best_idx = np.argmax(scores)
        if scores[batch_best_idx] > best_score:
            best_score = scores[batch_best_idx]
            best_shape = (int(types[batch_best_idx]), int(cx[batch_best_idx]),
                         int(cy[batch_best_idx]), int(hw[batch_best_idx]),
                         int(hh[batch_best_idx]), int(rot[batch_best_idx]))
            best_color = (int(colors[batch_best_idx][0]),
                         int(colors[batch_best_idx][1]),
                         int(colors[batch_best_idx][2]))

        remaining -= batch_n

    if best_shape is None:
        continue

    # Hill-climb the best candidate
    shape_type, b_cx, b_cy, b_hw, b_hh, b_rot = best_shape
    shape_type, f_cx, f_cy, f_hw, f_hh, f_rot, final_score, final_color = hill_climb(
        target_gpu, canvas_gpu, target_arr, canvas, shape_type, b_cx, b_cy, b_hw, b_hh, b_rot,
        best_score, best_color, mutated_samples, img_h, img_w, posterize_levels, rng
    )

    # Paint shape on canvas
    fr, fg, fb = final_color
    paint_shape(canvas, shape_type, f_cx, f_cy, f_hw, f_hh, f_rot, fr, fg, fb, img_h, img_w)

    # Update GPU canvas after painting
    if USE_GPU:
        canvas_gpu = cp.asarray(canvas, dtype=cp.float32)

    # Record shape in output format
    if shape_type == RECTANGLE:
        shape_data = [f_cx, f_cy, f_hw * 2, f_hh * 2]
    else:  # ROTATED_ELLIPSE
        shape_data = [f_cx, f_cy, f_hw, f_hh, f_rot]

    generated_shapes.append({
        "type": shape_type,
        "data": shape_data,
        "color": [fr, fg, fb, 255],  # forceOpaqueShapes=true
        "score": round(final_score / (img_w * img_h * 3 * 255.0 * 255.0), 6),
    })

    # Update progress bar postfix with speed and error info
    elapsed = time.time() - start_time
    speed = shape_idx / elapsed if elapsed > 0 else 0
    error_metric = final_score / (img_w * img_h) if img_w * img_h > 0 else 0
    pbar.set_postfix(speed=f"{speed:.1f} shapes/s", error=f"{error_metric:.4f}")

    # Checkpoint saving
    if shape_idx in save_at_points:
        checkpoint_data = build_output_json(generated_shapes, img_w, img_h)
        checkpoint_filename = f"checkpoint_{shape_idx}.json"
        checkpoint_path = os.path.join(OUTPUT_DIR, checkpoint_filename)
        with open(checkpoint_path, "w", encoding="utf-8") as f:
            json.dump(checkpoint_data, f)
        pbar.write(T["checkpoint_saved"].format(checkpoint_path, shape_idx))

# Free GPU memory
if USE_GPU:
    del target_gpu, canvas_gpu
    cp.get_default_memory_pool().free_all_blocks()

elapsed = time.time() - start_time
speed = stop_at / elapsed if elapsed > 0 else 0
print(T["generation_complete"].format(stop_at, elapsed, speed))


In [ ]:
#@title Output & Save
import json
from PIL import Image as PILImage
from IPython.display import display

print(T["saving_output"])

# Build final output JSON
output_data = build_output_json(generated_shapes, img_w, img_h)

# Save final JSON
image_basename = os.path.splitext(os.path.basename(image_path))[0]
output_filename = f"{image_basename}_geometry.json"
output_path = os.path.join(OUTPUT_DIR, output_filename)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output_data, f, indent=2)

print(T["output_saved"].format(output_path))
print(f"  Total shapes: {len(output_data['shapes'])}")
print(f"  File size: {os.path.getsize(output_path) / 1024:.1f} KB")

# Display preview of the generated result
print(f"\n{T['preview_title']}")
preview_image = PILImage.fromarray(canvas)
display(preview_image)

# Also save preview image
preview_path = os.path.join(OUTPUT_DIR, f"{image_basename}_preview.png")
preview_image.save(preview_path)
print(f"  Preview saved to: {preview_path}")
